**MGMT298D: Science and Strategy of AI**

# Week 6B: Building a Tiny LLM from Scratch

In this notebook we build a small GPT-style language model using Keras transformer blocks and train it on Yelp restaurant reviews. The goal is to watch the model learn: first producing gibberish, then plausible words, and finally coherent sentences as training progresses.

# 1 Setup

#### Import libraries for building and training a causal (autoregressive) transformer.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from datasets import load_dataset
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

---
# 2 Load & Prepare the Yelp Review Dataset

#### Load Yelp reviews from HuggingFace and tokenize them with a Keras TextVectorization layer. Each training example is a sequence of tokens; the target is the same sequence shifted by one position (next-token prediction).

In [ ]:
# Load Yelp review dataset
dataset = load_dataset('yelp_review_full', split='train')
print(f"Total Yelp reviews: {len(dataset):,}")

# Use a manageable subset
NUM_REVIEWS = 25000
texts = dataset['text'][:NUM_REVIEWS]

# Show a few examples
print(f"\nUsing {NUM_REVIEWS:,} reviews for training\n")
for i in range(3):
    stars = dataset['label'][i] + 1  # labels are 0-4, reviews are 1-5 stars
    preview = texts[i][:120].replace('\n', ' ')
    print(f"  [{stars}★] {preview}...")

In [ ]:
# Tokenize with Keras TextVectorization
VOCAB_SIZE = 10000
SEQ_LEN = 64  # context window — how many tokens the model sees at once

vectorizer = layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_sequence_length=SEQ_LEN + 1  # +1 so we can split into input/target
)
vectorizer.adapt(texts)

vocab = vectorizer.get_vocabulary()
print(f"Vocabulary size: {len(vocab):,} tokens")
print(f"Context window:  {SEQ_LEN} tokens")
print(f"\nSample vocabulary: {vocab[:20]}")
print(f"\nToken IDs for 'the food was great':")
print(f"  {vectorizer(['the food was great']).numpy()[0][:6]}")

In [ ]:
# Build training sequences: input = tokens[:-1], target = tokens[1:]
# This is the standard next-token prediction setup for language models

all_tokens = vectorizer(np.array(texts)).numpy()

# Filter out sequences that are mostly padding (too short)
valid_mask = np.sum(all_tokens > 0, axis=1) > 20
all_tokens = all_tokens[valid_mask]

x_train = all_tokens[:, :-1]  # input:  tokens 0 through SEQ_LEN-1
y_train = all_tokens[:, 1:]   # target: tokens 1 through SEQ_LEN

print(f"Training sequences: {x_train.shape[0]:,}")
print(f"Input shape:  {x_train.shape}  (batch, sequence)")
print(f"Target shape: {y_train.shape}  (batch, sequence)")
print(f"\nExample — first 10 input tokens:  {x_train[0][:10]}")
print(f"Example — first 10 target tokens: {y_train[0][:10]}")
print(f"\nNotice the target is the input shifted by one position.")
print(f"The model learns: given these tokens so far, predict the next one.")

---
# 3 Build the Tiny LLM

#### Define a causal (GPT-style) transformer. The key difference from the classification transformer in Week 6B's original version: we use a **causal attention mask** so each position can only attend to previous positions — the model can't cheat by looking ahead.

In [ ]:
class CausalTransformerBlock(layers.Layer):
    """Transformer block with causal (look-ahead) masking for autoregressive generation."""
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim // num_heads
        )
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation='relu'),
            layers.Dense(embed_dim),
        ])
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(rate)
        self.drop2 = layers.Dropout(rate)

    def call(self, inputs, training):
        # Causal mask: each token can only attend to itself and earlier tokens
        attn = self.att(inputs, inputs, use_causal_mask=True)
        attn = self.drop1(attn, training=training)
        out1 = self.norm1(inputs + attn)
        ffn = self.ffn(out1)
        ffn = self.drop2(ffn, training=training)
        return self.norm2(out1 + ffn)


class TokenAndPositionEmbedding(layers.Layer):
    """Combines learned token embeddings with learned positional embeddings."""
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)
        return self.token_emb(x) + self.pos_emb(positions)


print("Defined: CausalTransformerBlock (masked self-attention + FFN)")
print("Defined: TokenAndPositionEmbedding (token + positional)")

In [ ]:
# Model hyperparameters
EMBED_DIM = 128
NUM_HEADS = 4
FF_DIM = 256
NUM_BLOCKS = 2

def build_tiny_llm():
    inputs = layers.Input(shape=(SEQ_LEN,))
    x = TokenAndPositionEmbedding(SEQ_LEN, VOCAB_SIZE, EMBED_DIM)(inputs)
    for _ in range(NUM_BLOCKS):
        x = CausalTransformerBlock(EMBED_DIM, NUM_HEADS, FF_DIM, rate=0.1)(x)
    # Output: predict probability distribution over vocabulary at each position
    outputs = layers.Dense(VOCAB_SIZE, activation='softmax')(x)
    return keras.Model(inputs=inputs, outputs=outputs)

model = build_tiny_llm()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f"\nTiny LLM Architecture:")
print(f"  Embedding dim:    {EMBED_DIM}")
print(f"  Attention heads:  {NUM_HEADS}")
print(f"  Feed-forward dim: {FF_DIM}")
print(f"  Transformer blocks: {NUM_BLOCKS}")
print(f"  Context window:   {SEQ_LEN} tokens")
print(f"  Total parameters: {model.count_params():,}")
print(f"\nFor comparison, GPT-2 has 124M parameters. Ours has {model.count_params()/1e6:.1f}M.")

---
# 4 Text Generation Function

#### Define a function that generates text token by token. At each step the model predicts the next token distribution and we sample from it (with temperature control). This is exactly how ChatGPT and other LLMs generate text.

In [ ]:
def generate_text(model, prompt, max_tokens=60, temperature=0.7):
    """Generate text autoregressively from a prompt."""
    # Tokenize the prompt
    token_ids = vectorizer([prompt]).numpy()[0]
    # Find where the actual tokens end (before padding)
    nonzero = np.where(token_ids > 0)[0]
    if len(nonzero) == 0:
        generated = []
    else:
        generated = list(token_ids[:nonzero[-1] + 1])

    for _ in range(max_tokens):
        # Pad/truncate to SEQ_LEN for the model
        padded = np.zeros(SEQ_LEN, dtype='int32')
        start = max(0, len(generated) - SEQ_LEN)
        seq = generated[start:]
        padded[:len(seq)] = seq

        # Predict next token
        preds = model.predict(padded[np.newaxis, :], verbose=0)[0]
        next_pos = min(len(generated), SEQ_LEN - 1)
        next_token_probs = preds[next_pos]

        # Apply temperature and sample
        next_token_probs = np.log(next_token_probs + 1e-10) / temperature
        next_token_probs = np.exp(next_token_probs) / np.sum(np.exp(next_token_probs))
        next_token = np.random.choice(len(next_token_probs), p=next_token_probs)

        if next_token == 0:  # padding token — stop
            break
        generated.append(next_token)

    # Convert token IDs back to words
    id_to_word = {i: w for i, w in enumerate(vocab)}
    words = [id_to_word.get(t, '?') for t in generated]
    return ' '.join(words)


def show_next_token_predictions(model, prompt, top_k=8):
    """Show the model's top-k predictions for the next token after a prompt."""
    token_ids = vectorizer([prompt]).numpy()[0]
    nonzero = np.where(token_ids > 0)[0]
    if len(nonzero) == 0:
        print("  (empty prompt)")
        return
    seq_len_actual = nonzero[-1] + 1

    padded = np.zeros(SEQ_LEN, dtype='int32')
    padded[:seq_len_actual] = token_ids[:seq_len_actual]

    preds = model.predict(padded[np.newaxis, :], verbose=0)[0]
    next_pos = min(seq_len_actual, SEQ_LEN - 1)
    probs = preds[next_pos]

    top_indices = np.argsort(probs)[-top_k:][::-1]
    id_to_word = {i: w for i, w in enumerate(vocab)}

    print(f"  Prompt: \"{prompt}\"")
    print(f"  Top-{top_k} next token predictions:")
    for idx in top_indices:
        word = id_to_word.get(idx, '?')
        print(f"    {word:15s}  {probs[idx]:.4f}")
    print()


# Test prompts we'll use throughout
TEST_PROMPTS = [
    "the food was",
    "i would definitely",
    "the service at this restaurant",
    "we waited for"
]

print("Defined: generate_text() and show_next_token_predictions()")
print(f"Test prompts: {TEST_PROMPTS}")

---
# 5 Phase 1 — Undertrained Model (2 Epochs)

#### Train for just 2 epochs and observe the output. The model has barely seen the data, so its predictions will be nearly random — next-token probabilities spread across many unrelated words, and generated text is incoherent.

In [ ]:
PHASE1_EPOCHS = 2

print(f"=== Phase 1: Training for {PHASE1_EPOCHS} epochs ===")
print(f"The model has random weights — let's see what it produces.\n")

history_p1 = model.fit(
    x_train, y_train,
    batch_size=128,
    epochs=PHASE1_EPOCHS,
    validation_split=0.05
)

all_history = {
    'loss': list(history_p1.history['loss']),
    'val_loss': list(history_p1.history['val_loss']),
    'accuracy': list(history_p1.history['accuracy']),
    'val_accuracy': list(history_p1.history['val_accuracy'])
}

In [ ]:
# Next-token predictions after Phase 1
print("=== Phase 1: Next-Token Predictions ===\n")
for prompt in TEST_PROMPTS:
    show_next_token_predictions(model, prompt)

In [ ]:
# Text generation after Phase 1
print("=== Phase 1: Generated Text (2 epochs) ===\n")
for prompt in TEST_PROMPTS:
    generated = generate_text(model, prompt, max_tokens=40, temperature=0.8)
    print(f"  Prompt:    \"{prompt}\"")
    print(f"  Generated: {generated}")
    print()

print("Observation: The model produces mostly incoherent word salad.")
print("It may have learned a few common words, but has no sense of grammar or meaning.")

---
# 6 Phase 2 — Partially Trained Model (+8 Epochs = 10 Total)

#### Train for 8 more epochs (10 total). The model should start picking up common Yelp review patterns — food adjectives, service descriptions, and restaurant vocabulary. Predictions become more focused and generation starts to resemble real sentences.

In [ ]:
PHASE2_EPOCHS = 8

print(f"=== Phase 2: Training for {PHASE2_EPOCHS} more epochs (10 total) ===")

history_p2 = model.fit(
    x_train, y_train,
    batch_size=128,
    epochs=PHASE2_EPOCHS,
    validation_split=0.05
)

all_history['loss'].extend(history_p2.history['loss'])
all_history['val_loss'].extend(history_p2.history['val_loss'])
all_history['accuracy'].extend(history_p2.history['accuracy'])
all_history['val_accuracy'].extend(history_p2.history['val_accuracy'])

In [ ]:
# Next-token predictions after Phase 2
print("=== Phase 2: Next-Token Predictions ===\n")
for prompt in TEST_PROMPTS:
    show_next_token_predictions(model, prompt)

In [ ]:
# Text generation after Phase 2
print("=== Phase 2: Generated Text (10 epochs) ===\n")
for prompt in TEST_PROMPTS:
    generated = generate_text(model, prompt, max_tokens=40, temperature=0.8)
    print(f"  Prompt:    \"{prompt}\"")
    print(f"  Generated: {generated}")
    print()

print("Observation: The model now uses relevant Yelp vocabulary.")
print("Sentences are starting to have local coherence, but may drift off-topic.")

---
# 7 Phase 3 — More Trained Model (+15 Epochs = 25 Total)

#### Train for 15 more epochs (25 total). With enough training, the tiny model should produce reasonably coherent Yelp-style text — complete thoughts about food quality, service, and recommendations.

In [ ]:
PHASE3_EPOCHS = 15

print(f"=== Phase 3: Training for {PHASE3_EPOCHS} more epochs (25 total) ===")

history_p3 = model.fit(
    x_train, y_train,
    batch_size=128,
    epochs=PHASE3_EPOCHS,
    validation_split=0.05
)

all_history['loss'].extend(history_p3.history['loss'])
all_history['val_loss'].extend(history_p3.history['val_loss'])
all_history['accuracy'].extend(history_p3.history['accuracy'])
all_history['val_accuracy'].extend(history_p3.history['val_accuracy'])

In [ ]:
# Next-token predictions after Phase 3
print("=== Phase 3: Next-Token Predictions ===\n")
for prompt in TEST_PROMPTS:
    show_next_token_predictions(model, prompt)

In [ ]:
# Text generation after Phase 3
print("=== Phase 3: Generated Text (25 epochs) ===\n")
for prompt in TEST_PROMPTS:
    generated = generate_text(model, prompt, max_tokens=60, temperature=0.7)
    print(f"  Prompt:    \"{prompt}\"")
    print(f"  Generated: {generated}")
    print()

print("Observation: The model now generates more coherent Yelp-style text.")
print("It has learned common review patterns, food vocabulary, and sentence structure.")
print("Still far from perfect — but the improvement from Phase 1 is dramatic.")

---
# 8 Training Progress & Comparison

#### Visualize how loss and accuracy evolved across all three training phases. The decreasing loss corresponds to the improving text quality we observed above.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(all_history['loss']) + 1)

# Loss plot
ax1.plot(epochs, all_history['loss'], 'b-o', markersize=3, label='Train Loss')
ax1.plot(epochs, all_history['val_loss'], 'r-o', markersize=3, label='Val Loss')
ax1.axvline(x=2, color='gray', linestyle='--', alpha=0.7, label='Phase 1→2')
ax1.axvline(x=10, color='gray', linestyle=':', alpha=0.7, label='Phase 2→3')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy plot
ax2.plot(epochs, all_history['accuracy'], 'b-o', markersize=3, label='Train Accuracy')
ax2.plot(epochs, all_history['val_accuracy'], 'r-o', markersize=3, label='Val Accuracy')
ax2.axvline(x=2, color='gray', linestyle='--', alpha=0.7, label='Phase 1→2')
ax2.axvline(x=10, color='gray', linestyle=':', alpha=0.7, label='Phase 2→3')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (next-token)')
ax2.set_title('Next-Token Prediction Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Tiny LLM Training Progress Across All Phases', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\nPhase 1 (epoch  2): loss={all_history['loss'][1]:.4f}, accuracy={all_history['accuracy'][1]:.4f}")
print(f"Phase 2 (epoch 10): loss={all_history['loss'][9]:.4f}, accuracy={all_history['accuracy'][9]:.4f}")
print(f"Phase 3 (epoch 25): loss={all_history['loss'][-1]:.4f}, accuracy={all_history['accuracy'][-1]:.4f}")

---
# 9 Side-by-Side Comparison

#### The key takeaway: LLMs are just next-token predictors trained on lots of text. Even this tiny model (a fraction of GPT's size) learns to produce coherent restaurant reviews with enough training. Scale up the model, the data, and the compute — and you get ChatGPT.

In [ ]:
print("="*70)
print("SUMMARY: HOW TRAINING TRANSFORMS A LANGUAGE MODEL")
print("="*70)

print(f"\nModel: {model.count_params():,} parameters ({NUM_BLOCKS} transformer blocks)")
print(f"Data:  {x_train.shape[0]:,} Yelp review sequences")
print(f"Vocab: {VOCAB_SIZE:,} tokens, context window: {SEQ_LEN}")

print(f"\n{'Phase':<12s} {'Epochs':<10s} {'Loss':<12s} {'Accuracy':<12s} {'Text Quality'}")
print("-" * 70)
print(f"{'Phase 1':<12s} {'2':<10s} {all_history['loss'][1]:<12.4f} {all_history['accuracy'][1]:<12.4f} {'Random word salad'}")
print(f"{'Phase 2':<12s} {'10':<10s} {all_history['loss'][9]:<12.4f} {all_history['accuracy'][9]:<12.4f} {'Yelp vocabulary, partial coherence'}")
print(f"{'Phase 3':<12s} {'25':<10s} {all_history['loss'][-1]:<12.4f} {all_history['accuracy'][-1]:<12.4f} {'Coherent review-like text'}")
print("-" * 70)

print("\nKey Insights:")
print("  1. LLMs learn by predicting the next token — that's the entire training objective.")
print("  2. Early in training, the model is essentially guessing randomly.")
print("  3. With more training, it learns word co-occurrence patterns (\"food\" → \"was\" → \"good\").")
print("  4. Eventually it captures higher-level structure: sentence flow, sentiment, and style.")
print("  5. GPT-4 uses the same principle — just much larger models, more data, and more compute.")